In [4]:
# ==============================================================================
# 2026改修版: GPR_Linear (設計基準 A〜K 準拠)
# - 重要度: 係数ベクトルの絶対値 |beta| を適用 (SVM_Linearと同一の解釈性)
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(kernlab)
})

set.seed(42)

# -------------------------------
# 設定：パスと対象ファイル
# -------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
file_names <- c(
  "m_all_OH_rebuilt.csv"  # 今回の指定条件: 500S+OH (n_base_OH)
)

# 出力先設定
# 出力先をサーバー上の指定パスに変更
out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_GPR_Linear"
model_name <- "GPR_Linear_Coef"
run_dir <- file.path(out_root, paste0("Corr_1000_", model_name))
if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)

# 除外変数設定
inappropriate_vars <- c(
  'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
  'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
  'IonIoffF', 'DRMS', 'Var.1'
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPERS
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  return(as.numeric(r^2))
}

# (J) GPR Linear 係数算出関数
calc_gpr_linear_coefficients <- function(model, features) {
  # gaussprモデルからアルファ(重み)を取得
  alpha_mat <- alpha(model$finalModel) 
  # 訓練データを取得
  x_matrix <- as.matrix(model$trainingData[, features])
  
  # ベイズ線形回帰の係数 beta = X^T * alpha を計算
  beta <- t(x_matrix) %*% alpha_mat
  
  data.frame(
    Feature = features,
    Importance = as.numeric(abs(beta)), # SVMと同様に絶対値で評価
    Coefficient = as.numeric(beta)      # 正負の方向性保持用
  ) %>% arrange(desc(Importance))
}

# -------------------------------
# MAIN LOOP
# -------------------------------
summary_list <- list()
importance_list <- list()

for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) next
  
  df_raw <- tryCatch(read.csv(filepath, row.names = 1, check.names = FALSE), error = function(e) NULL)
  if (is.null(df_raw)) next

  for (target in target_vars) {
    if (target != "Jsc") next # 今回の指定: Jsc のみ
    if (!target %in% colnames(df_raw)) next

    # クレンジング
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) next

    # 変数除外
    features <- setdiff(colnames(df_temp), target_vars)
    features <- setdiff(features, inappropriate_vars)
    features <- features[!grepl(paste(physical_exclude_patterns, collapse="|"), features)]
    
    # ゼロ分散除外
    vars <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]

    # 多重共線性対策 (Corr > 0.99999)
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      cor_mat[is.na(cor_mat)] <- 0
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }

    # スケーリング [-1, 1]
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1

    # CV10設定
    train_ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")

    # 学習
    model <- tryCatch(
      train(
        x = df_scaled[, features, drop = FALSE], y = df_scaled[[target]],
        method = "gaussprLinear", trControl = train_ctrl, tuneGrid = NULL, metric = "RMSE"
      ),
      error = function(e) { return(NULL) }
    )
    if (is.null(model)) next

    # --- 保存処理 ---
    tag <- paste0(tools::file_path_sans_ext(file), "_", target)
    saveRDS(model, file.path(run_dir, paste0(tag, "_model.rds")))

    # CV予測
    pred_oof <- model$pred %>% 
      mutate(SampleID = rownames(df_scaled)[rowIndex], Type = "CV10_OOF") %>%
      select(SampleID, Observed = obs, Predicted = pred, Type)
    write.csv(pred_oof, file.path(run_dir, paste0(tag, "_pred_OOF.csv")), row.names = FALSE)

    # (J) 重要度: 係数絶対値に基づく算出
    imp_df <- calc_gpr_linear_coefficients(model, features)
    imp_df$File <- file
    imp_df$Target <- target
    importance_list[[length(importance_list)+1]] <- imp_df

    # サマリー
    summary_list[[length(summary_list)+1]] <- data.frame(
      File=file, Target=target, 
      CV10_R2=safe_r2(pred_oof$Observed, pred_oof$Predicted),
      CV10_RMSE=rmse(pred_oof$Observed, pred_oof$Predicted), 
      n_samples=nrow(df_scaled),
      n_features=length(features), 
      Params="LinearCoefAbs"
    )
    
    cat(sprintf("  Processed: %s - %s | CV10_R2: %.3f\n", file, target, tail(summary_list, 1)[[1]]$CV10_R2))
  }
}

# 出力
if (length(summary_list) > 0) {
  write.csv(do.call(rbind, summary_list), file.path(run_dir, "summary_GPR_Linear_Coef.csv"), row.names = FALSE)
  write.csv(do.call(rbind, importance_list), file.path(run_dir, "importance_GPR_Linear_Coef.csv"), row.names = FALSE)
}

cat("\n[DONE] GPR_Linear (Coefficient Analysis) completed.\n")

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."


  Processed: m_all_OH_rebuilt.csv - Jsc | CV10_R2: 0.741

[DONE] GPR_Linear (Coefficient Analysis) completed.
